In [1]:
import pandas as pd
import numpy as np
import re


In [2]:
# Open our data + basic info

# utf-8 causes decoding error
df = pd.read_csv('C:/my pet projects/shark_attacks/data/raw/shark_attacks_raw.csv',encoding='latin1')
print(df.shape)
print(df.info())    

(25723, 24)
<class 'pandas.DataFrame'>
RangeIndex: 25723 entries, 0 to 25722
Data columns (total 24 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Case Number             8702 non-null   str    
 1   Date                    6302 non-null   str    
 2   Year                    6300 non-null   float64
 3   Type                    6298 non-null   str    
 4   Country                 6252 non-null   str    
 5   Area                    5847 non-null   str    
 6   Location                5762 non-null   str    
 7   Activity                5758 non-null   str    
 8   Name                    6092 non-null   str    
 9   Sex                     5737 non-null   str    
 10  Age                     3471 non-null   str    
 11  Injury                  6274 non-null   str    
 12  Fatal (Y/N)             5763 non-null   str    
 13  Time                    2948 non-null   str    
 14  Species                 3464 non-null

In [3]:
# Percent of empty per column
missing_percent = df.isnull().mean().sort_values(ascending=False) * 100
print("Percentage of missing values per column:")
print(missing_percent)

Percentage of missing values per column:
Unnamed: 22               99.996112
Unnamed: 23               99.992225
Time                      88.539439
Species                   86.533453
Age                       86.506240
Sex                       77.697003
Activity                  77.615364
Location                  77.599813
Fatal (Y/N)               77.595926
Area                      77.269370
Name                      76.316915
Country                   75.694903
Injury                    75.609377
Investigator or Source    75.566614
Type                      75.516075
Year                      75.508300
href formula              75.504412
Date                      75.500525
Case Number.1             75.500525
pdf                       75.500525
Case Number.2             75.500525
href                      75.500525
original order            75.473312
Case Number               66.170353
dtype: float64


In [4]:
# Delete empty columns
df = df.drop(columns=['Unnamed: 22', 'Unnamed: 23'])

#Accuracy down to the day of the month is unlikely to be necessary; we'll single out the month, as statistics on attacks depending on the season may be of interest
df['Month'] = pd.to_datetime(df['Date'], errors='coerce').dt.month

# Delete date
df = df.drop(columns=['Date'])

print(f"After deleting empty columns: {df.shape}")

After deleting empty columns: (25723, 22)


In [5]:
necessary_columns = ['Fatal (Y/N)','Month', 'Case Number','Country','Area', 'Type', 'Activity']

# Delete rows where all the main characteristics are empty

df = df.dropna(subset=necessary_columns, how='all')
print(f"After deleting rows with empty main characteristics: {df.shape}")

# Percent of empty per column after cleaning
print(df.isnull().mean().sort_values(ascending=False) * 100)

After deleting rows with empty main characteristics: (8703, 22)
Time                      66.126623
Species                   60.197633
Age                       60.117201
Month                     45.248765
Sex                       34.080202
Activity                  33.838906
Location                  33.792945
Fatal (Y/N)               33.781455
Area                      32.816270
Name                      30.001149
Country                   28.162703
Injury                    27.909916
Investigator or Source    27.783523
Type                      27.634149
Year                      27.611169
href formula              27.599678
href                      27.588188
pdf                       27.588188
Case Number.2             27.588188
Case Number.1             27.588188
original order            27.507756
Case Number                0.011490
dtype: float64


In [6]:
# Fatal go from Y/N represent to binar 1/0 
def fatal_to_binary(value):
    if pd.isna(value):
        return None
    if value == "Y":
        return 1
    if value == "N":
        return 0
    else:
        return None

df['Fatal'] = df['Fatal (Y/N)'].apply(fatal_to_binary)


print(df[['Fatal (Y/N)', 'Fatal']].head(10))
print(df['Fatal'].value_counts(dropna=False))

# Get rid of Fatal(Y/N) because now we have clear binary view
df = df.drop(columns=['Fatal (Y/N)'])


  Fatal (Y/N)  Fatal
0           N    0.0
1           N    0.0
2           N    0.0
3           N    0.0
4           N    0.0
5           N    0.0
6           Y    1.0
7           N    0.0
8           N    0.0
9           N    0.0
Fatal
0.0    4293
NaN    3022
1.0    1388
Name: count, dtype: int64


In [7]:
# Rename column 'Species ' to 'Species'
df = df.rename(columns={'Species ': 'Species'})



# Sex to binary where f = 1, m = 0

def sex_to_binary(value):
    if pd.isna(value):
        return None
    val = str(value).lower()
    if "f" in val:
        return 1
    if "m" in val:
        return 0
    else:
        return None

df['Sex'] = df['Sex '].apply(sex_to_binary)


print(df[['Sex', 'Sex ']].head(10))
print(df['Sex'].value_counts(dropna=False))

# Get rid of Fatal(Y/N) because now we have clear binary view
df = df.drop(columns=['Sex '])


   Sex Sex 
0  1.0    F
1  1.0    F
2  0.0    M
3  0.0    M
4  0.0    M
5  0.0    M
6  0.0    M
7  0.0    M
8  0.0    M
9  0.0    M
Sex
0.0    5096
NaN    2970
1.0     637
Name: count, dtype: int64


In [8]:
# text columns to make lower case plus clear excess spaces
text_columns = ['Case Number', 'Type', 'Country', 'Area', 'Location', 'Activity', 'Name', 'Injury', 'Species', "Time"]

for column in text_columns:
        # Convert to string, lowercase, strip whitespace
        df[column] = df[column].astype(str).str.lower().str.strip()
        # Replace common empty string representations with NaN
        df[column] = df[column].replace(['nan', 'none', '', 'unknown'], None)


In [9]:
# Turning time to Time category


# getting hour from different time formats with numbers in them

def to_military_time(hour,ampm):
    if 'a' in ampm:
        if hour != 12:
            return hour
        else:
            return 0
    else:
        if hour != 12:
            return hour + 12
        else:
            return 12

def extract_hour(value):
    if pd.isna(value):
        return None
    value = str(value).lower().strip()
    # Pattern for getting an hour from formats that include numbers in them
    patterns = [
        (r'(\d{1,2})(?:h|:)(?:\d{2})?\s*(am|pm|a\.m\.|p\.m\.)', True),
        (r'(\d{1,2})\s*(am|pm|a\.m\.|p\.m\.)', True),
        (r'(\d{1,2})(?:h|:|\d{2})', False),
        (r'(\d{1})(\d{2})', False)      
    ]
    for pattern, has_ampm in patterns:
        match = re.search(pattern, value)
        if match:
            hour = int(match.group(1))
            
            if has_ampm:
                ampm = match.group(2).lower()
                hour = to_military_time(hour, ampm)

            # Is time corret?
            if 0 <= hour <= 23:
                return hour 
    
    return None

# Working with words meaning time in Time column
def categorize_by_words(value):
    if pd.isna(value):
        return None
    value = str(value).lower().strip()
    keywords = {
        'late night': ['after midnight', 'before dawn', 'wee hours', 'early morning hours'],
        'dawn': ['dawn', 'daybreak', 'sunrise', 'first light', 'before sunrise'],
        'morning': ['morning', 'am', 'a.m.', 'early morning', 'late morning', 'mid morning', 'mid-morning'],
        'afternoon': ['afternoon', 'pm', 'p.m.', 'after noon', 'mid afternoon', 'early afternoon', 'late afternoon'],
        'midday': ['midday', 'noon', 'lunchtime', 'just before noon', 'just after 12h00', '12h00'],
        'evening': ['evening', 'dusk', 'sunset', 'early evening', 'just before dusk', 'after dusk', 'nightfall'],
        'night': ['night', 'dark', 'after dark', 'late night', 'midnight']
    }

    for category, words in keywords.items():
        for word in words:
            if word in value:
                return category
    
    return None

# All in one function
def classify_time(value):
    if pd.isna(value):
        return None
    
    value = str(value).strip()
    
    keyword_category = categorize_by_words(value)
    if keyword_category:
        return keyword_category
    
    hour = extract_hour(value)

    if hour:
        if 4 <= hour < 7:
            return 'dawn'
        elif 7 <= hour < 12:
            return 'morning'
        elif 12 <= hour < 14:
            return 'midday'
        elif 14 <= hour < 17:
            return 'afternoon'
        elif 17 <= hour < 21:
            return 'evening'
        elif 21 <= hour < 24:
            return 'night'
        elif 0 <= hour < 4:
            return 'late night'
    
    return None

df['Time Category'] = df['Time'].apply(classify_time)

# Delete Time column
df = df.drop(columns=['Time'])

In [10]:
# Save cleaned dataset
df.to_csv('C:/my pet projects/shark_attacks/data/processed/shark_attacks_clean.csv', index=False)
print("\n=== Cleaned dataset saved to processed/ ===")


=== Cleaned dataset saved to processed/ ===
